# Préparation DPO — MediQAl, QCM (français)

Paires préférentielles (`prompt` / `chosen` / `rejected`) en français, à partir des QCM de
[ANR-MALADES/MediQAl](https://huggingface.co/datasets/ANR-MALADES/MediQAl) (CC-BY-4.0) : `mcqu` (une bonne réponse)
et `mcqm` (plusieurs).

- **`chosen`** : la réponse du corrigé, au format du SFT (« C. texte », ou une liste pour `mcqm`).
- **`rejected`** : la même réponse avec **une seule erreur**, construite à partir des distracteurs rédigés par les
  enseignants : l'erreur est plausible, et le corrigé garantit qu'elle est fausse. Préférences **validées par le
  corrigé**, sans juge LLM.
- **Cas distincts du SFT** : seuls les cas cliniques non retenus par les notebooks SFT v4 sont utilisés.
- **Orienté triage** : cas clinique obligatoire, priorité aux cas avec constantes et vocabulaire d'urgence
  (`projet_14.selection`), budget de questions et plafond par spécialité.
- **Même format que le SFT v4** : la question est vue avec l'historique de son cas (questions précédentes et leurs
  réponses de référence, `projet_14.conversation.history_prompt`).
- **Sortie** : `data/v4/dpo_mediqal_{train,validation,test}.jsonl`, puis la version anonymisée dans `data/v4/anonymized/`.

# Import

In [1]:
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer

from projet_14.conversation import CASE_LABEL, history_prompt, split_of
from projet_14.mediqal import LETTERS, OPTIONS_HEADER, answer_letters, case_id, clean_text, correct_letters, format_answer
from projet_14.rgpd import run
from projet_14.selection import select_cases, triage_flags

OUT = Path("../data/v4")
STEM = "dpo_mediqal"
SEED = 42
MODEL_NAME = "Qwen/Qwen3-1.7B-Base"  # pour compter les tokens avec le tokenizer du modèle entraîné

CONFIGS = {"mcqu": False, "mcqm": True}   # configuration -> réponses multiples
BUDGETS = {"mcqu": 750, "mcqm": 750}      # paires au plus par configuration, tous splits confondus (plafond)
MAX_SUBJECT_SHARE = 0.15                  # part maximale du budget par spécialité
MAX_PROMPT_CHARS = 2_800                  # historique raccourci au-delà (le cas reste)
MAX_LENGTH = 1_024                        # prompt + la plus longue des deux réponses, en tokens (DPO_MAX_LENGTH)
REVIEW_SAMPLE = 30                        # paires exportées pour relecture

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chargement

Les trois parties officielles de chaque configuration sont réunies : on refait le découpage par cas clinique.

In [2]:
df = pd.concat([
    pd.concat([part.to_pandas() for part in load_dataset("ANR-MALADES/MediQAl", config).values()]).assign(config=config)
    for config in CONFIGS
], ignore_index=True)
df.groupby("config").size()

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


config
mcqm    10617
mcqu    17017
dtype: int64

# Nettoyage et filtres

Mêmes règles que les notebooks SFT : espaces en trop, mention « (cochez la réponse juste) » retirée
(« réponse fausse » gardée : elle change le sens de la question), questions qui renvoient à une image écartées.
**Cas clinique obligatoire** : le triage part d'un patient décrit.

In [3]:
TEXT_COLUMNS = ["clinical_case", "question"] + [f"answer_{letter.lower()}" for letter in LETTERS]
for col in TEXT_COLUMNS:
    df[col] = df[col].map(clean_text, na_action="ignore")
df["clinical_case"] = df["clinical_case"].replace("", None)
df["question"] = df["question"].str.replace(re.compile(r"^\(\s*cochez la réponse juste\s*\)\s*", re.I), "", regex=True)

IMAGE = re.compile(r"ci-dessous|ci-contre|ci-joint|figure|l'image|cette image|photo|schéma suivant|document suivant|annexe", re.I)
image = df["question"].str.contains(IMAGE) | df["clinical_case"].fillna("").str.contains(IMAGE)
df = df[~image].dropna(subset=["clinical_case", "question", "correct_answers"])
df = df.drop_duplicates(subset=["config", "clinical_case", "question"])
df.groupby("config").size()

config
mcqm    3970
mcqu    6765
dtype: int64

# Répartition par cas et exclusion du SFT

Même identifiant et même split que les notebooks SFT (`split_of`). Les cas retenus par un notebook SFT v4 (`oeq`,
`mcqu` ou `mcqm`) sont exclus : le DPO travaille sur d'autres cas que ceux appris en SFT.

In [4]:
df["case_id"] = df["clinical_case"].map(case_id)
df["split"] = df["clinical_case"].map(split_of)
question_norm = df["question"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
df = df[~pd.concat([df["config"], df["case_id"], question_norm], axis=1).duplicated()]

sft_cases = set()
for stem in ("sft_mediqal_oeq", "sft_mediqal_mcqu", "sft_mediqal_mcqm"):
    for split in ("train", "validation", "test"):
        sft_cases |= set(pd.read_json(OUT / "anonymized" / f"{stem}_{split}.anonymized.jsonl", lines=True)["case_id"])
in_sft = df["case_id"].isin(sft_cases)
print(f"questions de cas retenus en SFT : {in_sft.sum()} écartées ({len(sft_cases)} cas)")
df = df[~in_sft]
pd.crosstab(df["config"], df["split"], margins=True)

questions de cas retenus en SFT : 2173 écartées (588 cas)


split,test,train,validation,All
config,,,,
mcqm,742,2137,305,3184
mcqu,1119,3681,578,5378
All,1861,5818,883,8562


# Orientation triage

Mêmes critères que le SFT (`projet_14.selection`) : score de 0 à 2 (**constantes** dans le cas, vocabulaire
d'**urgence** dans le cas, les questions ou le corrigé), puis budget par configuration, réparti entre les splits, et
plafond par spécialité. La sélection se fait par cas entier.

In [5]:
def correct_text(row) -> str:
    return " ".join(row[f"answer_{letter.lower()}"] for letter in correct_letters(row) if row[f"answer_{letter.lower()}"])


df["correct_text"] = df.apply(correct_text, axis=1)
cases = df.groupby(["config", "case_id"], sort=False).agg(
    split=("split", "first"), medical_subject=("medical_subject", "first"), n_questions=("question", "size"),
    case_text=("clinical_case", "first"),
    text=("question", lambda q: " ".join(q)), answers=("correct_text", lambda a: " ".join(a)),
).reset_index()
flags = [triage_flags(c, f"{c}\n{t}\n{a}", "fr") for c, t, a in zip(cases["case_text"], cases["text"], cases["answers"])]
cases = cases.assign(patient=True, constantes=[f[0] for f in flags], urgence=[f[1] for f in flags])
cases["score"] = cases["constantes"].astype(int) + cases["urgence"].astype(int)
print(pd.crosstab(cases["config"], cases["score"], values=cases["n_questions"], aggfunc="sum", margins=True), "\n")

selected = pd.concat([
    select_cases(cases[cases["config"] == config], BUDGETS[config], MAX_SUBJECT_SHARE, group="medical_subject")
    for config in CONFIGS
])
selected.groupby(["config", "split"]).agg(cas=("case_id", "size"), questions=("n_questions", "sum"),
                                          constantes=("constantes", "mean"), urgence=("urgence", "mean")).round(2)

score      0     1    2   All
config                       
mcqm    1687  1453   44  3184
mcqu    2448  2633  297  5378
All     4135  4086  341  8562 



cas  questions  constantes  urgence
config split                                          
mcqm   test         41        150        0.39     0.76
       train       197        525        0.37     0.65
       validation   23         75        0.39     0.78
mcqu   test         32        150        0.59     0.84
       train       138        525        0.44     0.70
       validation   18         75        0.56     0.89

# Construction des paires

Pour chaque question des cas retenus, dans l'ordre du récit :

- **prompt** : consigne, cas, questions précédentes avec leurs réponses de référence, puis la question ;
- **`chosen`** : les bonnes propositions du corrigé ;
- **`rejected`** : une seule erreur, parmi celles qui respectent la consigne. On prend le type le moins utilisé
  jusqu'ici (au hasard en cas d'égalité) : omissions (`chosen` plus longue) et ajouts (`chosen` plus courte) restent
  en nombre proche, et la longueur ne trahit pas la bonne réponse. En v3, le DPO avait appris à « cocher moins ».

| Type | Construction | Possible si |
|---|---|---|
| `substitution` | une bonne proposition remplacée par une mauvaise | une mauvaise proposition existe |
| `omission` (`mcqm`) | une bonne proposition retirée | 3 bonnes réponses ou plus (il en reste au moins deux) |
| `ajout` (`mcqm`) | une mauvaise proposition ajoutée | 4 bonnes réponses au plus |

En `mcqu`, seule la substitution garde « une seule proposition » : la `rejected` est un distracteur.
Les propositions restent dans l'ordre A à E : la position ne trahit pas l'erreur.

In [6]:
rng = random.Random(SEED)
type_counts = Counter()  # erreurs déjà construites, par configuration : pour équilibrer les types
df["order"] = pd.to_numeric(df["id"], errors="coerce")
keep = df.merge(selected[["config", "case_id"]], on=["config", "case_id"]).sort_values("order")


def options_of(row) -> dict:
    return {letter: row[f"answer_{letter.lower()}"] for letter in LETTERS if row[f"answer_{letter.lower()}"]}


def rejected_letters(gold: list[str], wrong: list[str], multiple: bool, config: str) -> tuple[str, list[str]] | None:
    # Type d'erreur et lettres de la rejected, ou None si aucune erreur n'est possible
    types = []
    if wrong:
        types.append("substitution")
    if multiple and len(gold) >= 3:
        types.append("omission")
    if multiple and wrong and len(gold) <= 4:
        types.append("ajout")
    if not types:
        return None
    # Type le moins utilisé jusqu'ici parmi les possibles : omissions (chosen plus longue) et ajouts (chosen plus
    # courte) en nombre proche, pour que la longueur ne trahisse pas la bonne réponse (« cocher moins », v3)
    fewest = min(type_counts[config, kind] for kind in types)
    kind = rng.choice([kind for kind in types if type_counts[config, kind] == fewest])
    type_counts[config, kind] += 1
    removed = rng.choice(gold)  # bonne proposition retirée (substitution, omission)
    added = rng.choice(wrong) if wrong else None  # mauvaise proposition ajoutée (substitution, ajout)
    if kind == "substitution":
        letters = [g for g in gold if g != removed] + [added]
    elif kind == "omission":
        letters = [g for g in gold if g != removed]
    else:
        letters = gold + [added]
    return kind, sorted(letters)


rows = []
for (config, cid), group in keep.groupby(["config", "case_id"], sort=False):
    multiple = CONFIGS[config]
    records = group.to_dict("records")
    turns = [{"question": r["question"], "options": options_of(r), "multiple": multiple,
              "answer": format_answer(r, correct_letters(r), as_list=multiple)} for r in records]
    for k, r in enumerate(records):
        gold = correct_letters(r)
        wrong = [letter for letter in options_of(r) if letter not in gold]
        error = rejected_letters(gold, wrong, multiple, config)
        if error is None:
            continue
        rows.append({
            "id": str(r["id"]), "config": config, "source": f"MediQAl-{config}", "case_id": cid, "split": r["split"],
            "medical_subject": r["medical_subject"], "golden_answer": r["correct_answers"], "rejected_type": error[0],
            "prompt": history_prompt(r["clinical_case"], turns, k, MAX_PROMPT_CHARS),
            "chosen": turns[k]["answer"], "rejected": format_answer(r, error[1], as_list=multiple),
        })
pairs = pd.DataFrame(rows).merge(selected[["config", "case_id", "constantes", "urgence", "score"]], on=["config", "case_id"])
pairs = pairs.assign(pair_check="golden_answer", triage=pairs["score"] >= 1, lang="fr")
pd.crosstab([pairs["source"], pairs["rejected_type"]], pairs["split"], margins=True)

split                       test  train  validation   All
source       rejected_type                               
MediQAl-mcqm ajout            49    179          23   251
             omission         48    171          30   249
             substitution     53    175          22   250
MediQAl-mcqu substitution    150    525          75   750
All                          300   1050         150  1500

# Filtre sur la longueur (tokens)

Prompt + la plus longue des deux réponses ≤ `MAX_LENGTH` tokens : au-delà, TRL couperait la fin des réponses.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def n_tokens(texts: list[str]) -> list[int]:
    return [len(ids) for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]]


prompt_tokens = n_tokens(["\n".join(m["content"] for m in p) for p in pairs["prompt"]])
pairs = pairs.assign(n_prompt=[n + 5 * len(p) for n, p in zip(prompt_tokens, pairs["prompt"])],  # + balises du gabarit
                     n_chosen=n_tokens(list(pairs["chosen"])), n_rejected=n_tokens(list(pairs["rejected"])))
fits = pairs["n_prompt"] + pairs[["n_chosen", "n_rejected"]].max(axis=1) <= MAX_LENGTH
print(f"{len(pairs)} paires, {(~fits).sum()} trop longues")
pairs = pairs[fits]
pairs["chosen_longer"] = pairs["n_chosen"] > pairs["n_rejected"]
pairs.groupby(["source", "rejected_type"])["chosen_longer"].agg(["size", "mean"]).round(2)

1500 paires, 27 trop longues


size  mean
source       rejected_type            
MediQAl-mcqm ajout           251  0.00
             omission        247  1.00
             substitution    249  0.45
MediQAl-mcqu substitution    726  0.45

In [8]:
# Une paire, telle que la verra le modèle
example = pairs[pairs["prompt"].map(len) >= 4].iloc[0]
for message in example["prompt"]:
    print(f"--- {message['role']}\n{message['content'][:500]}\n")
print(f"--- CHOSEN\n{example['chosen']}\n--- REJECTED ({example['rejected_type']})\n{example['rejected']}")

--- system
Tu es un assistant médical expert. Réponds de façon précise et concise.

--- user
Cas clinique : Monsieur CAM.., 55 ans, ouvrier agricole, consulte pour une masse latérocervicale droite apparue il y a trois mois. Pas de douleur, mais il signale une anorexie et une asthénie. Il fume plus d'un paquet de cigarettes par jour et avoue boire 2 litres de vin par jour. Il se plaint de dysphagie sans troubles respiratoires. Antécédents : bronchite chronique non traitée. L'examen montre : - Une pharyngite érythémateuse, un état buccodentaire très altéré, et une stase salivaire qui empê

--- assistant
- C. La stase salivaire
- D. L'immobilité laryngée
- E. La dysphagie

--- user
Question : Parmi les raisons suivantes, quelle (s) est(sont) celle(s) qui pourrai(en)t vous interdire la chirurgie :

Propositions :
A. L'altération trop importante de l'état général
B. Une tumeur de plus d'1 cm de diamètre
C. Une métastase pleuro-pulmonaire
D. Une adénopathie bilatérale
E. Une adénopathie de p

## Échantillon à relire

`REVIEW_SAMPLE` paires d'entraînement, tirées au hasard, exportées pour une relecture clinique (colonne
`preference_juste` : oui / non). Le fichier contient le texte avant anonymisation : il reste dans `data/`, non
versionné.

In [9]:
REVIEW_DIR = OUT / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
sample = pairs[pairs["split"] == "train"].sample(min(REVIEW_SAMPLE, len(pairs)), random_state=SEED)
review = sample[["case_id", "source", "rejected_type", "golden_answer"]].assign(
    question=sample["prompt"].map(lambda p: p[-1]["content"][:800]), chosen=sample["chosen"], rejected=sample["rejected"],
    preference_juste="", commentaire="")
review.to_csv(REVIEW_DIR / f"{STEM}_review.csv", index=False)
review.head(3)

,case_id,source,rejected_type,golden_answer,question,chosen,rejected,preference_juste,commentaire
543,2bc9e63aa0a0,MediQAl-mcqu,substitution,B,"Question : Parmi les éléments suivants, le(s)q...",B. Terrain atopique,C. Absence d'hyperleucocytose à l'hémogramme,,
702,46ac0a99e6ac,MediQAl-mcqm,substitution,"A,C","Question : En cours d'évolution, le patient pe...",- A. Leucémie aiguë myéloïde\n- C. Aplasie méd...,- A. Leucémie aiguë myéloïde\n- B. Leucose à p...,,
489,ff45b484ed3f,MediQAl-mcqu,substitution,D,Cas clinique : François est âgé de 3 ans. Il e...,D. Maintenir l'enfant en position assise,A. Assurer une vidange gastrique,,


# Export JSONL

In [10]:
METADATA = ["id", "source", "case_id", "medical_subject", "rejected_type", "pair_check", "golden_answer",
            "constantes", "urgence", "triage", "lang"]
for split in ("train", "validation", "test"):
    part = pairs[pairs["split"] == split].assign(
        chosen=lambda d: d["chosen"].map(lambda a: [{"role": "assistant", "content": a}]),
        rejected=lambda d: d["rejected"].map(lambda a: [{"role": "assistant", "content": a}]),
    )
    Dataset.from_pandas(part[METADATA + ["prompt", "chosen", "rejected"]], preserve_index=False).to_json(
        OUT / f"{STEM}_{split}.jsonl", force_ascii=False)
    print(split, len(part), "paires")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 34.13ba/s]

train 1027 paires


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 123.17ba/s]

validation 148 paires


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 75.58ba/s]

test 298 paires


# Anonymisation des données

Comme pour les QCM du SFT : Presidio sur les messages **user** du prompt, avant l'en-tête « Propositions : »
(`keep_after`). Le cas et les questions décrivent le patient ; les propositions, les réponses de l'historique,
`chosen` et `rejected` sont des énoncés rédigés par les enseignants.

In [11]:
reports = {
    split: run(OUT / f"{STEM}_{split}.jsonl", lang="fr", output_dir=OUT / "anonymized", roles=["user"], keep_after=OPTIONS_HEADER)
    for split in ("train", "validation", "test")
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,1027,252,482
validation,148,33,58
test,298,75,140


# Contrôle final

In [12]:
final = load_dataset("json", data_files={s: str(OUT / "anonymized" / f"{STEM}_{s}.anonymized.jsonl") for s in ("train", "validation", "test")})
print(final)

for split in final.values():
    for example in split:
        chosen, rejected = example["chosen"][0]["content"], example["rejected"][0]["content"]
        gold = frozenset(letter.strip().upper() for letter in example["golden_answer"].split(","))
        assert answer_letters(chosen) == gold and answer_letters(rejected) != gold, "préférence non conforme au corrigé"
        assert example["prompt"][1]["content"].startswith(CASE_LABEL), "prompt sans cas clinique"
        for m in example["prompt"][1:]:
            text = m["content"].split(OPTIONS_HEADER, 1)[-1] if m["role"] == "user" else m["content"]
            assert "<PERSON>" not in text + chosen + rejected, "proposition ou réponse modifiée"

# Aucun cas partagé entre les splits, ni avec le SFT
ids = {s: set(d["case_id"]) for s, d in final.items()}
assert not ids["train"] & (ids["validation"] | ids["test"]) and not ids["validation"] & ids["test"], "cas partagé entre splits"
assert not sft_cases & set().union(*ids.values()), "cas commun avec le SFT"
print("OK : préférences conformes au corrigé, propositions intactes, aucun cas partagé entre splits ni avec le SFT")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 1027 examples [00:00, 58610.11 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 148 examples [00:00, 27735.89 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 298 examples [00:00, 41402.58 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'rejected_type', 'pair_check', 'golden_answer', 'constantes', 'urgence', 'triage', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 1027
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'rejected_type', 'pair_check', 'golden_answer', 'constantes', 'urgence', 'triage', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 148
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'rejected_type', 'pair_check', 'golden_answer', 'constantes', 'urgence', 'triage', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 298
    })
})
OK : préférences conformes au corrigé, propositions intactes, aucun cas partagé entre splits ni avec le SFT


# Limites

- **Erreurs construites** : la `rejected` est une erreur plausible (un distracteur des enseignants), mais ce n'est pas
  une erreur réellement commise par le modèle. Un DPO sur les erreurs du modèle, validées par des soignants, reste
  la cible (roadmap).
- **Format QCM** : les paires portent sur le choix des propositions, pas sur des réponses rédigées.
- **Cas non retenus par le SFT** : les cas les plus proches du triage sont d'abord allés au SFT ; le DPO prend les
  suivants, avec les mêmes critères.